## 실습과제 1

- https://github.com/ancestor9/data/blob/main/customers.csv 을 읽어와서 데이터베이스를 만들고 user하는 table에 CRUD하는 코드를 만들어달라
-grdio로 외부 URL 로 웹서비스해라(CRUD)

In [ ]:
import pandas as pd
import sqlite3
import os
import gradio as gr

CUSTOMER_DB_NAME = "customer_color_db.db"

In [ ]:
def init_db():
    if os.path.exists(CUSTOMER_DB_NAME):
        os.remove(CUSTOMER_DB_NAME)
    conn = sqlite3.connect(CUSTOMER_DB_NAME, check_same_thread=False)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS customers (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            color TEXT NOT NULL
        )
    """)
    # 초기 데이터 삽입 (예시)
    sample_data = [("살사", "blue"), ("알레", "red"), ("니콜", "pink")]
    cursor.executemany("INSERT INTO customers (name, color) VALUES (?, ?)", sample_data)
    conn.commit()
    return conn

conn = init_db()

In [ ]:
def read_customers():
    df = pd.read_sql("SELECT * FROM customers", conn)
    df.columns = ["ID", "이름 (Name)", "색상 (Color)"]
    return df

def get_customer_dropdown_list():
    df = pd.read_sql("SELECT id, name FROM customers", conn)
    return [(f"[{row['id']}] {row['name']}", row['id']) for _, row in df.iterrows()]

def create_action(name, color):
    # 1. 빈 칸 검사
    if not name or not color:
        return "⚠️ 오류: 이름과 색상을 모두 입력해주세요.", read_customers(), gr.update(choices=get_customer_dropdown_list())

    # 2. 이름에 숫자 포함 여부 검사 (사용자 요청 사항)
    if any(char.isdigit() for char in name):
        return "❌ 오류: 이름에는 숫자를 입력할 수 없습니다!", read_customers(), gr.update(choices=get_customer_dropdown_list())

    try:
        cursor = conn.cursor()
        cursor.execute("INSERT INTO customers (name, color) VALUES (?, ?)", (name, color))
        conn.commit()
        return f"✅ 성공: '{name}' 데이터가 추가되었습니다.", read_customers(), gr.update(choices=get_customer_dropdown_list())
    except Exception as e:
        return f"❌ 오류: {e}", read_customers(), gr.update()

def delete_action(target_id):
    if not target_id:
        return "⚠️ 오류: 삭제할 항목을 선택해주세요.", read_customers(), gr.update()

    cursor = conn.cursor()
    cursor.execute("DELETE FROM customers WHERE id = ?", (target_id,))
    conn.commit()

    updated_dropdown = gr.update(choices=get_customer_dropdown_list(), value=None)
    return "✅ 성공: 데이터가 삭제되었습니다.", read_customers(), updated_dropdown

In [ ]:
custom_css = """
    /* 전체 배경 및 기본 설정 */
    body { background-color: #121212 !important; color: #e0e0e0 !important; }
    .gradio-container { background-color: #121212 !important; }

    /* 입력창 스타일 */
    input, select { border-radius: 8px !important; background-color: #1e1e1e !important; color: white !important; border: 1px solid #333 !important; }

    /* [수정] 추가 버튼 스타일 - 초록색 (Green) */
    .green-button {
        background-color: #22c55e !important; /* Tailwind Green 500 */
        color: white !important;
        border: none !important;
        border-radius: 8px !important;
        font-weight: bold !important;
    }
    .green-button:hover {
        background-color: #16a34a !important; /* 조금 더 어두운 초록색 */
    }

    /* 삭제 버튼 스타일 - 빨간색 (Red) */
    .red-button {
        background-color: #ef4444 !important;
        color: white !important;
        border: none !important;
        border-radius: 8px !important;
    }
"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("# 🎨  Customer Color Manager")

    # 입력 영역
    with gr.Row():
        with gr.Column():
            name_input = gr.Textbox(label="이름 ", placeholder="이름 입력")
            color_input = gr.Textbox(label="색상", placeholder="예: Blue, Red...")
            btn_add = gr.Button("➕ Add Record", variant="primary")

    gr.Markdown("---")

    # 출력 및 삭제 영역
    with gr.Row():
        with gr.Column(scale=2):
            table_output = gr.Dataframe(value=read_customers(), interactive=False)

        with gr.Column(scale=1):
            gr.Markdown("### Delete Section")
            delete_dropdown = gr.Dropdown(choices=get_customer_dropdown_list(), label="삭제할 항목 선택")
            btn_delete = gr.Button("🗑️ Delete Selected", variant="stop")

    status_msg = gr.Textbox(label="Status / Result")

    # 이벤트 연결
    btn_add.click(create_action, [name_input, color_input], [status_msg, table_output, delete_dropdown])
    btn_delete.click(delete_action, [delete_dropdown], [status_msg, table_output, delete_dropdown])

demo.launch(share=True)

/tmp/ipykernel_9978/602088021.py:30: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d209c5ec6bb2c3d0c2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
